# Lab 13 — Part 2: Search Functions and Retrieval Approaches

**Learning goals:** Reuse the retrieval pipeline in a function, explore different queries, and compare sparse and dense retrieval.

## Setup — Run this cell first

The cell below recreates the small document collection and TF-IDF vectors used in Part 1.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."
]

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)

## 1. Make a simple search function

Now combine the steps so you can search the document collection using any query.

In [2]:
def search(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    for i in top_ids:
        print(f"Score: {scores[i]:.3f} | {documents[i]}")

search("How can we measure similarity between text vectors?")

Score: 0.686 | Cosine similarity measures how similar two text vectors are.
Score: 0.148 | Named entity recognition identifies names, places, and organizations in text.


## 2. Sparse vs. dense retrieval (conceptual comparison)

| Sparse retrieval | Dense retrieval |
|---|---|
| Represents text using word-based features, e.g., TF-IDF | Represents text using learned embeddings |
| Works well when important query words appear in documents | Can identify similar meanings even when different words are used |
| Simple and lightweight | Usually requires a pretrained embedding model |

**Important limitation:** TF-IDF mainly matches words, not meanings. A query about *"automobiles"* may not retrieve a document about *"cars"* unless the words also overlap in other ways.

In modern RAG, retrieval can be sparse, dense, or a combination of both.

## 3. Does a high-ranked result always mean a good match?

The search function returns the **highest-scoring** documents, even when all the scores are zero. A high *rank* does not always mean a high *similarity score*.

For example, our collection is about NLP. What happens when we search for an unrelated topic like `"volcano eruption"`?

In [3]:
search("volcano eruption", k=2)

# Compare it with a relevant query
print("--- Relevant query ---")
search("cosine similarity", k=2)

Score: 0.000 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.
--- Relevant query ---
Score: 0.594 | Cosine similarity measures how similar two text vectors are.
Score: 0.000 | Information retrieval finds relevant documents for a user query.


## 4. Improve search: skip results with a score of zero

We can improve the function by printing only documents with a **positive similarity score**.

In [4]:
def search_filtered(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    found = False
    for i in top_ids:
        if scores[i] > 0:
            print(f"Score: {scores[i]:.3f} | {documents[i]}")
            found = True
    if not found:
        print("No matching documents found.")

search_filtered("volcano eruption")
print("---")
search_filtered("cosine similarity")

No matching documents found.
---
Score: 0.594 | Cosine similarity measures how similar two text vectors are.


## 5. Practice questions

1. Use `search()` to find a note about **named entity recognition**. Which document ranks first?
2. Add `"BERT is a transformer model that uses bidirectional context."` to `documents`, rebuild the TF-IDF vectors using `vectorizer.fit_transform(documents)`, and search for `"transformer models"`. Why was rebuilding necessary?

In [5]:
# Practice: run your own queries here
search("named entity recognition", k=2)

Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.


#### Answer — Practice 1 — document 3, score 0.625

Document **3**, *"Named entity recognition identifies names, places, and organizations in text."*,
ranks first: it holds all three query terms (`named`, `entity`, `recognition`) and no other
document holds any of them. The second line, at **0.000**, is the tie-break filler that
`search()` prints because `k=2` asks for two rows; `search_filtered()` from section 4 is the
version that drops it.

In [6]:
# Practice 2. Everything above works on the original six documents, so the new
# document is added last: refitting in place would leave the earlier cells'
# outputs describing a corpus that no longer exists.
print("before:", document_vectors.shape, "| vocabulary:", len(vectorizer.vocabulary_))
had_transformer = "transformer" in vectorizer.vocabulary_

documents.append("BERT is a transformer model that uses bidirectional context.")
document_vectors = vectorizer.fit_transform(documents)

print("after: ", document_vectors.shape, "| vocabulary:", len(vectorizer.vocabulary_))
print("'transformer' in the old vocabulary:", had_transformer)
print("'transformer' in the new vocabulary:", "transformer" in vectorizer.vocabulary_)
print()
search("transformer models", k=2)

before: (6, 39) | vocabulary: 39
after:  (7, 45) | vocabulary: 45
'transformer' in the old vocabulary: False
'transformer' in the new vocabulary: True

Score: 0.408 | BERT is a transformer model that uses bidirectional context.
Score: 0.000 | Cosine similarity measures how similar two text vectors are.


#### Answer — Practice 2 — why the vectors had to be rebuilt

`fit_transform()` does two things that a later `transform()` cannot: it **learns the vocabulary**
(one column per term) and it **computes the IDF weight of every term** from the current
documents. It also returns one row per document.

Nothing about that updates itself when the list grows. Against the old matrix the new document
has no row at all, and `transformer`, `bert`, `bidirectional`, `model` and `context` are not
columns. `transform()` drops out-of-vocabulary words silently — it does not error — so the query
`"transformer models"` would become an all-zero vector and every one of the six scores would be
0.000. Rebuilding with `fit_transform(documents)` extends the matrix to **7 × 45**, gives the new
sentence row 6, and re-estimates the IDF of every term now that a seventh document exists.

One detail worth noticing after the rebuild: document 0, *"Transformers use self-attention…"*,
still scores **0.000** for `"transformer models"`. The query token is `transformer`; the
document's token is `transformers`. `models` is out of vocabulary too, because the BERT sentence
says `model`. A word-matching retriever does not connect a plural to its singular — the same
limitation as the `document` / `documents` example in Part 1.